## Options Open Interest Spike

In [1]:
%load_ext autoreload
%autoreload 2
import logging
from qrt.data import *
from qrt.utils import *
from qrt.constants import *
from qrt.qrt_utils import *
from qrt.strategies import *

### Load Options Data

In [2]:
options = pd.read_parquet(WRDS_DIR / 'options' / '2025.parquet')
options.memory_usage(deep=True).sum() / 1024**2

np.float64(5995.286762237549)

### LSEG Prices

In [ ]:
EXCHCD_TO_RIC_SUFFIX = {
    1:  '.N',   # NYSE
    2:  '.A',   # NYSE American (AMEX)
    3:  '.OQ',  # NASDAQ
    4:  '.MW',  # Midwest
    13: '.N',   # NYSE (post-2003 codes)
    14: '.OQ',  # NASDAQ (alt code)
    19: '.A',   # AMEX (alt code)
}

# options = pd.read_parquet(WRDS_DIR / 'options' / '2025.parquet')
options = pd.concat(
    [pd.read_parquet(WRDS_DIR / 'options' / f'{y}.parquet') for y in [2024, 2025]], ignore_index=True
)
secid_map = pd.read_parquet(WRDS_DIR / 'secid_map.parquet')[['secid', 'permno', 'mktcap_mm']].drop_duplicates(subset='secid')
crsp = pd.read_parquet(WRDS_DIR / 'crsp_rua_daily.parquet')
active_lseg = pd.read_parquet(PRICE_DIR / LSEG_ACTIVE)

# Top 80 pctile mkt caps
# secid_map = secid_map[secid_map['mktcap_mm'] >= secid_map['mktcap_mm'].quantile(1 - 0.5)]

# Lseg returns
active_lseg['ret'] = active_lseg.groupby('RIC')['Close'].pct_change(fill_method=None)
active_lseg = active_lseg[['Date', 'ret', 'RIC', 'Close']].rename(columns={'Date':'date', 'RIC':'ric', 'Close':'price'})
active_lseg['date'] = pd.to_datetime(active_lseg['date'])

# Get ric to match to lseg
permo_ticker = crsp[['permno', 'ticker', 'exchcd']].drop_duplicates(keep='last')
permo_ticker['ric'] = permo_ticker['ticker'] + permo_ticker['exchcd'].map(EXCHCD_TO_RIC_SUFFIX)

# Combine all tables
options['date'] = pd.to_datetime(options['date'])
options['exdate'] = pd.to_datetime(options['exdate'])
options = options.merge(secid_map, on='secid').merge(
    permo_ticker, on=['permno'], how='left'
).merge(active_lseg, on=['ric', 'date'], how='left')

/var/folders/6p/48x6s06j165c8fyjy5kyz3cc0000gn/T/ipykernel_4006/2896135524.py:23:FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


### WRDS Prices

In [3]:
options = pd.read_parquet(WRDS_DIR / 'options' / '2025.parquet')
link = pd.read_parquet(WRDS_DIR / 'secid_map.parquet')[['secid', 'permno']].drop_duplicates(subset='secid')
crsp = pd.read_parquet(WRDS_DIR / 'crsp_rua_daily.parquet')

crsp['date'] = pd.to_datetime(crsp['date'])
crsp['ret'] = np.where(
    crsp['dlret'].notna(),
    (1 + crsp['ret'].fillna(0)) * (1 + crsp['dlret']) - 1,
    crsp['ret']
)
crsp = crsp.drop(columns='dlret').drop(columns=['volume'])

options['date'] = pd.to_datetime(options['date'])
options['exdate'] = pd.to_datetime(options['exdate'])
options = options.merge(link, on='secid').merge(crsp, on=['permno', 'date'], how='left')

### Signal Construction

In [4]:
# ── 1. Aggregate options into daily signals per stock ────────────────
opts = options.copy()
opts['mid'] = (opts['best_bid'] + opts['best_offer']) / 2
opts['dollar_vol'] = opts['volume'] * opts['mid'] * 100
opts['dte'] = (opts['exdate'] - opts['date']).dt.days

# Filter OTM only
otm = opts[
    ((opts['cp_flag'] == 'C') & (opts['delta'].between(0.05, 0.45))) |
    ((opts['cp_flag'] == 'P') & (opts['delta'].between(-0.45, -0.05)))
]

signal = otm.groupby(['permno', 'date']).agg(
    call_vol=('volume', lambda x: x[otm.loc[x.index, 'cp_flag'] == 'C'].sum()),
    put_vol=('volume', lambda x: x[otm.loc[x.index, 'cp_flag'] == 'P'].sum()),
    total_oi=('open_interest', 'sum'),
    total_dollar_vol=('dollar_vol', 'sum'),
    price=('price', 'first'),
    ret=('ret', 'first'),
    # mktcap_mm=('mktcap_mm', 'first'),
).reset_index()

signal['pc_ratio'] = signal['put_vol'] / signal['call_vol'].clip(lower=1)

signal = signal.sort_values(['permno', 'date'])
signal['oi_zscore'] = signal.groupby('permno')['total_oi'].transform(
    lambda x: (x - x.rolling(20).mean()) / x.rolling(20).std()
)

# ── 2. Forward returns ──────────────────────────────────────────────
signal['fwd_1d'] = signal.groupby('permno')['ret'].shift(-2)
signal['fwd_2d'] = signal.groupby('permno')['ret'].shift(-3)
signal['fwd_3d'] = signal.groupby('permno')['ret'].transform(
    lambda x: (1 + x.shift(-1)) * (1 + x.shift(-2)) * (1 + x.shift(-3)) - 1
)
signal['fwd_5d'] = signal.groupby('permno')['ret'].transform(
    lambda x: (1 + x.shift(-1)) * (1 + x.shift(-2)) * (1 + x.shift(-3)) *
              (1 + x.shift(-4)) * (1 + x.shift(-5)) - 1
)

# Drop warmup period (need 20 days for z-score)
signal = signal.dropna(subset=['oi_zscore', 'fwd_1d'])

# ── 3. Decile sort ────────────────────────────────────────────────
signal['oi_decile'] = signal.groupby('date')['oi_zscore'].transform(
    lambda x: pd.qcut(x.rank(method='first'), 10, labels=False) + 1
)
# # Ventiles (~24 stocks per bucket)
# signal['oi_decile'] = signal.groupby('date')['oi_zscore'].transform(
#     lambda x: pd.qcut(x.rank(method='first'), 20, labels=False) + 1
# )

### Evaluate Signal

In [5]:
import matplotlib.pyplot as plt

def evaluate_signal(signal, n_quantiles=10, cost_bps=2, fwd_ret='fwd_1d'):
    """Evaluate D10 long-only, beta-hedged, and L/S strategy."""
    d10 = signal[signal['oi_decile'] == n_quantiles]
    d1 = signal[signal['oi_decile'] == 1]
    
    d10_daily = d10.groupby('date')[fwd_ret].mean()
    d1_daily = d1.groupby('date')[fwd_ret].mean()
    mkt_daily = signal.groupby('date')[fwd_ret].mean()
    ls = (d10_daily - d1_daily).dropna().sort_index()
    
    # Beta-hedged: D10 - beta * market
    lo = d10_daily.dropna().sort_index()
    both = pd.DataFrame({'d10': lo, 'mkt': mkt_daily}).dropna()
    beta = both['d10'].cov(both['mkt']) / both['mkt'].var()
    hedged = (both['d10'] - beta * both['mkt']).sort_index()
    
    # Stocks per leg
    n_long = d10.groupby('date')['permno'].nunique().mean()
    n_short = d1.groupby('date')['permno'].nunique().mean()
    
    # Turnover
    holdings = d10.groupby('date')['permno'].apply(set)
    dates = sorted(holdings.index)
    turnover = [len(holdings[dates[i]] - holdings[dates[i-1]]) / len(holdings[dates[i]]) 
                for i in range(1, len(dates))]
    avg_turnover = np.mean(turnover)
    cost_drag = avg_turnover * (cost_bps / 10000) * 252
    
    # Stats
    lo_sharpe = lo.mean() / lo.std() * np.sqrt(252)
    lo_cum = (1 + lo).cumprod()
    lo_dd = ((lo_cum / lo_cum.cummax()) - 1).min()
    
    ls_sharpe = ls.mean() / ls.std() * np.sqrt(252)
    ls_cum = (1 + ls).cumprod()
    ls_dd = ((ls_cum / ls_cum.cummax()) - 1).min()
    
    h_sharpe = hedged.mean() / hedged.std() * np.sqrt(252)
    h_cum = (1 + hedged).cumprod()
    h_dd = ((h_cum / h_cum.cummax()) - 1).min()
    
    # Monthly
    ls.index = pd.to_datetime(ls.index)
    monthly = ls.groupby(ls.index.to_period('M')).apply(lambda x: (1+x).prod()-1)
    
    # Benchmark
    start = str(lo_cum.index.min().date())
    end = str(lo_cum.index.max().date())
    bm = get_history(['.SPX'], start=start, end=end).iloc[:, 0].dropna()
    bm.index = pd.to_datetime(bm.index)
    bm = bm.reindex(lo_cum.index, method='ffill')
    bm_cum = (1 + bm.pct_change().dropna()).cumprod()
    bm_cum = bm_cum.reindex(lo_cum.index).ffill()
    
    print(f"=== D10 Long Only ({n_long:.0f} stocks/day) ===")
    print(f"Sharpe:           {lo_sharpe:.2f}")
    print(f"Annual:           {lo.mean() * 252 * 100:.1f}%")
    print(f"Max drawdown:     {lo_dd*100:.1f}%")
    print(f"Win rate:         {(lo > 0).mean():.1%}")
    
    print(f"\n=== D10 Beta-Hedged (beta={beta:.2f}) ===")
    print(f"Sharpe:           {h_sharpe:.2f}")
    print(f"Annual:           {hedged.mean() * 252 * 100:.1f}%")
    print(f"Max drawdown:     {h_dd*100:.1f}%")
    print(f"Win rate:         {(hedged > 0).mean():.1%}")
    
    print(f"\n=== D10-D1 Long/Short ({n_long:.0f}L / {n_short:.0f}S) ===")
    print(f"Sharpe:           {ls_sharpe:.2f}")
    print(f"Annual:           {ls.mean() * 252 * 100:.1f}%")
    print(f"Max drawdown:     {ls_dd*100:.1f}%")
    print(f"Win rate:         {(ls > 0).mean():.1%}")
    print(f"Daily turnover:   {avg_turnover:.1%}")
    print(f"Cost drag:        {cost_drag*100:.1f}% ann")
    print(f"Net annual:       {(ls.mean()*252 - cost_drag)*100:.1f}%")
    print(f"Positive months:  {(monthly > 0).sum()}/{len(monthly)}")
    print(f"\nMonthly returns:")
    print(monthly.to_string())
    
    # Plot
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    
    axes[0].plot(lo_cum.index, lo_cum, label=f'Long Only (SR {lo_sharpe:.2f})')
    axes[0].plot(h_cum.index, h_cum, label=f'Beta-Hedged (SR {h_sharpe:.2f})')
    axes[0].plot(ls_cum.index, ls_cum, label=f'L/S (SR {ls_sharpe:.2f})')
    axes[0].plot(bm_cum.index, bm_cum, label='SPX', linestyle='--', color='gray')
    axes[0].set_ylabel('Cumulative Return')
    axes[0].legend()
    axes[0].set_title('OI Spike Momentum')
    axes[0].grid(True, alpha=0.3)
    
    ls_drawdown = ls_cum / ls_cum.cummax() - 1
    axes[1].fill_between(ls_drawdown.index, ls_drawdown, 0, color='red', alpha=0.3)
    axes[1].set_ylabel('Drawdown')
    axes[1].set_xlabel('Date')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return ls, monthly

ls, monthly = evaluate_signal(signal)
print()

/Users/dcunning/Code/Python/imperial/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860:RuntimeWarning: Mean of empty slice.
/Users/dcunning/Code/Python/imperial/.venv/lib/python3.12/site-packages/numpy/_core/_methods.py:144:RuntimeWarning: invalid value encountered in scalar divide


ParserError: Unknown string format: NaT

### Testing

In [ ]:
active_lseg = pd.read_parquet(PRICE_DIR / LSEG_ACTIVE)
active_prices_rua, active_vol_rua = get_timeseries(active_lseg, stock_index=RUA), get_timeseries(active_lseg, value='Volume', stock_index=RUA)
active_eligible_rua = eligible_to_trade(active_prices_rua, active_vol_rua, stock_index=RUA)
cols = active_eligible_rua.columns[active_eligible_rua.iloc[-1]]


rua = permo_ticker.copy()
# ric_uni = pd.read_csv(DATA_DIR / LSEG_ACTIVE_CONSTITUENTS_FILE).RIC
ric_uni = list(active_prices_rua[cols].columns)
permnos = rua[rua.ric.isin(ric_uni)].permno.to_list()
permno_str = ",".join([str(int(p)) for p in permnos])

db = get_connection()

data = db.raw_sql(f"""
        SELECT a.permno, b.ticker, b.ncusip,
               abs(a.prc) * a.shrout / 1000 as mktcap_mm
        FROM crsp.dsf AS a
        INNER JOIN crsp.dsenames AS b
            ON a.permno = b.permno
            AND a.date BETWEEN b.namedt AND b.nameendt
        WHERE a.date = (SELECT max(date) FROM crsp.dsf)
          AND a.permno IN ({permno_str})
          AND b.shrcd IN (10, 11)
          AND a.prc IS NOT NULL
          AND a.shrout IS NOT NULL
        ORDER BY mktcap_mm DESC
    """)
data.to_csv(DATA_DIR / 'tmp.csv', index=False)


Connecting to WRDS...
Loading library list...
Done
Connected.



,permno,ticker,ncusip,mktcap_mm
0,14593,AAPL,03783310,3785304.39566
1,86580,NVDA,67066G10,3288761.8551
2,10107,MSFT,59491810,3133802.3415
3,84788,AMZN,02313510,2306888.26329
4,93436,TSLA,88160R10,1296350.6304
...,...,...,...,...
1574,24511,RR,76550410,149.7771
1575,89244,WW,98262P10,101.42728
1576,92594,SLS,81642T20,73.19728
1577,92486,FLNT,34380C20,42.53004


In [10]:
data.to_csv(DATA_DIR / 'tmp.csv', index=False)